In [1]:
!pip install xgboost

In [2]:
import sys


print(sys.executable)

C:\Users\irg\AppData\Local\Programs\Python\Python313\python.exe


In [3]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.pipeline import Pipeline
from  sklearn.compose import ColumnTransformer

from sklearn.linear_model import LinearRegression , Lasso, LassoCV, Ridge
from sklearn.preprocessing import OneHotEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor , GradientBoostingRegressor
# from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import BaggingRegressor

from sklearn.metrics import accuracy_score , r2_score , mean_squared_error

In [4]:
data = pd.read_csv("../data/processed/customer_features.csv")

In [5]:
data.head()

,CustomerID,recency_days,purchase_frequency,historical_revenue,total_quantity,unique_products,customer_tenure_days,average_order_value,avg_quantity_per_order,orders_per_month,revenue_90d,revenue_30d,previous_90d_revenue,spending_trend,future_revenue,country
0,12346,133,1,77183.60,74215,1,133,77183.600000,74215.000000,0.184049,0.00,0.0,77183.60,-77183.60,0.00,United Kingdom
1,12347,54,3,1823.43,1117,63,175,607.810000,372.333333,0.439024,636.25,0.0,1187.18,-550.93,2486.57,Iceland
2,12348,56,3,1487.24,2124,22,166,495.746667,708.000000,0.459184,367.00,0.0,1120.24,-753.24,310.00,Finland
3,12350,118,1,334.40,197,17,118,334.400000,197.000000,0.202703,0.00,0.0,334.40,-334.40,0.00,Norway
4,12352,70,5,1561.81,254,26,104,312.362000,50.800000,1.119403,280.66,0.0,1281.15,-1000.49,944.23,Norway


In [6]:
data = data.drop(columns=["CustomerID"],axis=1)

In [7]:
data.head()

,recency_days,purchase_frequency,historical_revenue,total_quantity,unique_products,customer_tenure_days,average_order_value,avg_quantity_per_order,orders_per_month,revenue_90d,revenue_30d,previous_90d_revenue,spending_trend,future_revenue,country
0,133,1,77183.60,74215,1,133,77183.600000,74215.000000,0.184049,0.00,0.0,77183.60,-77183.60,0.00,United Kingdom
1,54,3,1823.43,1117,63,175,607.810000,372.333333,0.439024,636.25,0.0,1187.18,-550.93,2486.57,Iceland
2,56,3,1487.24,2124,22,166,495.746667,708.000000,0.459184,367.00,0.0,1120.24,-753.24,310.00,Finland
3,118,1,334.40,197,17,118,334.400000,197.000000,0.202703,0.00,0.0,334.40,-334.40,0.00,Norway
4,70,5,1561.81,254,26,104,312.362000,50.800000,1.119403,280.66,0.0,1281.15,-1000.49,944.23,Norway


In [8]:
X = data.drop("future_revenue",axis=1)
Y = data["future_revenue"]

In [9]:
num_cols = X.select_dtypes(include=["int64","float64"]).columns.tolist()

In [10]:
cat_cols = X.select_dtypes(include=["object","category"]).columns.tolist()

In [11]:
num_cols

['recency_days',
 'purchase_frequency',
 'historical_revenue',
 'total_quantity',
 'unique_products',
 'customer_tenure_days',
 'average_order_value',
 'avg_quantity_per_order',
 'orders_per_month',
 'revenue_90d',
 'revenue_30d',
 'previous_90d_revenue',
 'spending_trend']

In [12]:
data[num_cols]

,recency_days,purchase_frequency,historical_revenue,total_quantity,unique_products,customer_tenure_days,average_order_value,avg_quantity_per_order,orders_per_month,revenue_90d,revenue_30d,previous_90d_revenue,spending_trend
0,133,1,77183.60,74215,1,133,77183.600000,74215.000000,0.184049,0.00,0.00,77183.60,-77183.60
1,54,3,1823.43,1117,63,175,607.810000,372.333333,0.439024,636.25,0.00,1187.18,-550.93
2,56,3,1487.24,2124,22,166,495.746667,708.000000,0.459184,367.00,0.00,1120.24,-753.24
3,118,1,334.40,197,17,118,334.400000,197.000000,0.202703,0.00,0.00,334.40,-334.40
4,70,5,1561.81,254,26,104,312.362000,50.800000,1.119403,280.66,0.00,1281.15,-1000.49
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2713,33,2,980.54,788,47,54,490.270000,394.000000,0.714286,980.54,0.00,0.00,980.54
2714,65,1,51.00,20,1,65,51.000000,20.000000,0.315789,51.00,0.00,0.00,51.00
2715,85,1,180.60,45,10,85,180.600000,45.000000,0.260870,180.60,0.00,0.00,180.60
2716,8,5,535.05,336,142,145,107.010000,67.200000,0.857143,217.15,99.47,317.90,-100.75


In [13]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [14]:
num_pipeline = Pipeline([
    ("scaler" , StandardScaler())
])

cat_pipeline = Pipeline([
    ("encoder",OneHotEncoder(drop="first",handle_unknown="ignore"))
])

In [15]:
preprocessor = ColumnTransformer([
    ("num_pipeline",num_pipeline,num_cols),
    ("cat_pipeline",cat_pipeline,cat_cols)
])


In [16]:
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

models = [
    LinearRegression(),
    Lasso(),
    Ridge(),
    RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    GradientBoostingRegressor(
        n_estimators=200,
        random_state=42
    )
]

accuracy =[]
for model in models:
    model.fit(X_train_scaled,Y_train)
    pred = model.predict(X_test_scaled)

    acc = r2_score(Y_test,pred)

    print(f"Model = {model} , Accuracy = {acc}")
    accuracy.append(acc)

C:\Users\irg\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\irg\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.041e+09, tolerance: 1.488e+07
  model = cd_fast.enet_coordinate_descent(


Model = LinearRegression() , Accuracy = 0.33243937236654564
Model = Lasso() , Accuracy = 0.3445780860974762
Model = Ridge() , Accuracy = 0.3563714640922744
Model = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42) , Accuracy = -0.33656117291586307
Model = GradientBoostingRegressor(n_estimators=200, random_state=42) , Accuracy = -1.4862403993328885


In [17]:
model = RandomForestRegressor(random_state=42)
param = {
    'n_estimators':[200,150,100,250],
    'max_depth':[None,4,7,9,15,17],
    'min_samples_split':[23,25,30,39]    
}
gridcv = GridSearchCV(estimator=model,param_grid=param,cv = 5,n_jobs=-1,scoring="r2")

In [ ]:
gridcv.fit(X_train_scaled,Y_train)